In [ ]:
import logging
from turtle import pd
import sys, os
sys.path.append("../..")
from src.pipelines.raw.kinexon import (
    get_detected_events_for_fixture as kinexon_get_detected_events_for_fixture,
    get_positions_for_session as kinexon_get_positions_for_session,
    get_sessions_for_team as kinexon_get_sessions_for_team,
    get_teams_for_season as kinexon_get_teams_for_season,
)
from src.pipelines.raw.sportradar import (
    get_competition_id as sr_get_competition_id,
    get_season_id as sr_get_season_id,
    get_teams_for_season as sr_get_teams_for_season,
    get_fixtures_for_season as sr_get_fixtures_for_season,
    get_fixture_events as sr_get_fixture_events,
    get_players_for_fixture as sr_get_players_for_fixture,
)
from src.pipelines.normalized.match_detected_shots import normalize_match_detected_shots
from src.pipelines.normalized.match_positions import normalize_match_positions
from src.pipelines.normalized.match_events import normalize_match_events
from src.pipelines.normalized.match_events import normalize_match_events_goals, normalize_match_events_setup
from src.pipelines.normalized.match_players import normalize_match_players

from src.pipelines.synced.players import extract_players_for_match
from src.pipelines.synced.shot_events import sync_shot_events, render_shot_event

from src.pipelines.normalized.matches import normalize_matches

from src.pipelines.features.calc_xg_features import calculate_xg_features

from dotenv import load_dotenv
from src.hbl_etl_dagster.utils.api_helper import (
    get_api_kinexon,
    get_api_sportradar,
)
import pandas as pd

In [ ]:
# api init
api_sportradar = get_api_sportradar()
api_kinexon = get_api_kinexon()

YEAR = 2025
COMPETITION_NAME = "1. Handball-Bundesliga"

POSITIONS_CACHE_PATH = "../../data/positions/"


logging.basicConfig(level=logging.WARNING)

In [ ]:
# Fetch competition and season IDs
competition_id = sr_get_competition_id(api_sportradar, COMPETITION_NAME)
season_id = sr_get_season_id(api_sportradar, competition_id, YEAR)
print(f"Using competition_id: {competition_id} for {COMPETITION_NAME},\nseason_id: {season_id} for year {YEAR}")

In [ ]:
# Fetch teams and fixtures for the season
teams_sportradar = sr_get_teams_for_season(api_sportradar, season_id)
display(teams_sportradar.head(18))

In [ ]:
fixtures = sr_get_fixtures_for_season(api_sportradar, season_id)
display(fixtures)

In [ ]:
# keep only those fixtures that are not in the future
from datetime import datetime
print(f"Total fixtures fetched: {len(fixtures)}")
fixtures = fixtures[fixtures["startTimeLocal"] <= datetime.now().isoformat()]
print(f"Fixtures until now: {len(fixtures)}")
# sort by startTimeLocal
fixtures = fixtures.sort_values(by="startTimeLocal").reset_index(drop=True)
# rename fixtureId to fixture_id for consistency
fixtures = fixtures.rename(columns={"fixtureId": "fixture_id"})
display(fixtures)

In [ ]:
# kinexon teams for the season
teams_kinexon = kinexon_get_teams_for_season(api_kinexon, str(YEAR))
print(f"Kinexon teams for season {YEAR}:")
print(teams_kinexon)

In [ ]:
start_time_min = pd.to_datetime(fixtures["startTimeLocal"]).min()
end_time_max = pd.to_datetime(fixtures["startTimeLocal"]).max()
df_team_sessions = kinexon_get_sessions_for_team(
    api=api_kinexon,
    team_id=0,
    start_date=start_time_min,
    end_date=end_time_max,
)
display(df_team_sessions)



In [ ]:
df_normalized_matches = normalize_matches(
    df_fixtures_sportradar_raw=fixtures,
    df_sessions_kinexon_raw=df_team_sessions,
    # df_teams_sportradar_raw=teams_sportradar_raw,
    # df_teams_kinexon_raw=teams_kinexon_raw,
)
display(df_normalized_matches)

# display only rows with nan in them
print("Not matched fixtures details:")
df_matched_details = df_normalized_matches[df_normalized_matches.isna().any(axis=1)]
display(df_matched_details)

In [ ]:
# # fuzzy match sessions to fixtures by using "description" and "nameLocal" and the date
# from thefuzz import process

# def match_sessions_to_fixtures(sessions: pd.DataFrame, fixtures: pd.DataFrame) -> pd.DataFrame:
#     fixture_names = fixtures["nameLocal"].tolist()
#     matched_fixtures = []
#     for _, session in sessions.iterrows():
#         session_desc = session["description"]
#         if pd.isna(session_desc):
#             continue
#         best_match, score = process.extractOne(session_desc, fixture_names)
#         if score >= 80:  # threshold for a good match
#             matched_fixture = fixtures[fixtures["nameLocal"] == best_match].iloc[0]
#             matched_fixtures.append({
#                 "session_id": session["id"],
#                 "fixture_id": matched_fixture["fixtureId"],
#                 "match_score": score
#             })
#     return pd.DataFrame(matched_fixtures)

# df_matched_sessions_fixtures = match_sessions_to_fixtures(df_team_sessions, fixtures)

# # stats how many sessions were matched
# total_sessions = len(df_team_sessions)
# matched_sessions = len(df_matched_sessions_fixtures)
# print(f"Total sessions: {total_sessions}, Matched sessions: {matched_sessions}, Match rate: {matched_sessions/total_sessions:.2%}")
# # unmatched sessions
# unmatched_sessions = df_team_sessions[~df_team_sessions["id"].isin(df_matched_sessions_fixtures["session_id"])]
# print("Unmatched sessions:")
# display(unmatched_sessions)
# # display matched sessions with fixture details
# df_matched_details = df_matched_sessions_fixtures.merge(
#     fixtures,
#     left_on="fixture_id",
#     right_on="fixtureId",
#     how="left"
# )
# print("Matched sessions with fixture details:")
# display(df_matched_details)

In [ ]:
for _, fixture in df_normalized_matches.iterrows():
    fixture_id = fixture["fixture_id"]
    fixture_gameday = fixture["round_number"]
    fixture_name = fixture["description"]
    fixture_start_time = fixture["start_time_local"]
    session_id = int(fixture["session_id"]) if not pd.isna(fixture["session_id"]) else None
    print(
        f"Gameday {fixture_gameday} | Time {fixture_start_time} | Name {fixture_name} | SessionId {session_id} | Processing fixture ID: {fixture_id}"
    )
    if fixture_name is None:
        continue
    # check if flensburg is home, else skip
    if "Flensburg" not in fixture_name.split("vs")[0]:
        print("Skipping fixture as Flensburg is not home.")
        continue

    # Fetch fixture events
    fixture_events = sr_get_fixture_events(api_sportradar, fixture_id)
    # get players
    players = sr_get_players_for_fixture(api_sportradar, fixture_events)

    # fetch kinexon events
    detected_events = kinexon_get_detected_events_for_fixture(
        api_kinexon, session_id
    )
    path_file_positions = os.path.join(
        POSITIONS_CACHE_PATH, f"kinexon_positions_{session_id}.0.parquet.gzip"
    )

    # check if file already exist
    if os.path.exists(path_file_positions):
        print(
            f"File kinexon_positions_{session_id}.0.parquet.gzip already exists, loading."
        )
        positions = pd.read_parquet(path_file_positions)
    else:
        # remove once tested
        continue
        # fetch kinexon positions
        positions = kinexon_get_positions_for_session(api_kinexon, session_id)
        # for positions insert session_id
        positions["session_id"] = session_id
        positions["fixture_id"] = fixture_id
        # save as kinexon_positions_{'session_id'}.0.parquet.gzip
        positions.to_parquet(
            f"kinexon_positions_{session_id}.0.parquet.gzip",
            compression="gzip",
        )

    detected_events = kinexon_get_detected_events_for_fixture(
        api_kinexon, session_id
    )
    # insert fixture_id
    detected_events["fixture_id"] = fixture_id
    positions["session_id"] = session_id
    positions["fixture_id"] = fixture_id
    fixture_events["fixture_id"] = fixture_id
    players["fixture_id"] = fixture_id

    # display(detected_events)
    # # Normalize data
    normalized_detected_events = normalize_match_detected_shots(
        detected_events
    )
    normalized_positions = normalize_match_positions(positions)
    normalized_events = normalize_match_events(fixture_events)
    normalized_setup = normalize_match_events_setup(normalized_events)
    normalized_goals = normalize_match_events_goals(normalized_events)
    normalized_players = normalize_match_players(players)

    # funny hack for flensburg. if flensburg is in the team home, there is a y offset of 12.5 m
    if fixture["team_name_home"] == "SG Flensburg-Handewitt":
        print("Applying Flensburg Y offset hack.")
        normalized_positions.loc[:, "y_m"] = normalized_positions["y_m"] - 12.5

    # sync data
    synced_players = extract_players_for_match(
        pd.DataFrame(fixture).T,
        normalized_setup,
        normalized_detected_events,
        normalized_positions,
        normalized_players,
    )
    print("Synced players:")
    display(synced_players)

    synced_shot_events = sync_shot_events(
        df_match_normalized=pd.DataFrame(fixture).T,
        df_match_events_normalized_goals=normalized_goals,
        df_match_detected_shots_normalized=normalized_detected_events,
        df_positions_normalized=normalized_positions,
        df_players=synced_players,
    )
    print(f"Synced shot events. Total goal events: {len(normalized_goals)}")
    display(synced_shot_events)

    rendered = render_shot_event(
            df_goals=synced_shot_events,
            df_positions=normalized_positions,
            max_events=5,
            require_throw_ts=False,
            output_dir="../../output/shot_event_renders/",
        )

    df_xg_features = calculate_xg_features(
        df_match_normalized=pd.DataFrame(fixture).T,
        df_shot_events=synced_shot_events,
        df_positions_normalized=normalized_positions,
    )
    print("XG Features:")
    display(df_xg_features)
    # break